# Full-Scale Silver Transformation

Transforms the full Bronze smart-meter dataset into a validated, typed and deduplicated Silver layer in Microsoft Fabric.

## Purpose

- Read the full Bronze Delta dataset
- Standardise and type analytical fields
- Apply data-quality validation rules
- Separate invalid records for audit
- Deduplicate household-timestamp business keys
- Derive analytical date and time attributes
- Persist validated and rejected Silver tables
- Reconcile all source, rejected, duplicate and Silver record counts

> The validation rules developed against the development sample are applied here to the full 167.9M-record Bronze dataset.

## 1. Load Full Bronze Dataset

Read the Delta-backed Bronze table containing the complete landed smart-meter dataset and its source-lineage metadata.

In [2]:
# ----------------------------------------
# Project 05 - Microsoft Fabric Analytics Platform
# Notebook 06 - Full Silver Transformation
# ----------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.table("bronze.meter_readings")

print("Full-scale Silver transformation initialised.")

StatementMeta(, fa3f41bb-f9fc-4f93-aee4-527020762ac0, 4, Finished, Available, Finished, False)

Full-scale Silver transformation initialised.


## 2. Standardise and Type Source Fields

Convert the raw Bronze fields into the consistent analytical schema required by the Silver layer.

In [3]:
standardised_df = (
    bronze_df
    .select(
        F.col("LCLid").alias("HouseholdID"),
        F.col("stdorToU").alias("TariffType"),
        F.col("DateTime").alias("ReadingTimestampRaw"),
        F.col("ConsumptionKWhRaw").alias("ConsumptionKWhRaw"),
        F.col("SourceFileName"),
        F.col("SourceFilePath"),
        F.col("IngestionTimestamp"),
        F.col("SourceSystem")
    )
)

StatementMeta(, fa3f41bb-f9fc-4f93-aee4-527020762ac0, 5, Finished, Available, Finished, False)

In [4]:
typed_df = (
    standardised_df
    .withColumn(
        "ReadingTimestamp",
        F.to_timestamp("ReadingTimestampRaw")
    )
    .withColumn(
        "ConsumptionKWh",
        F.col("ConsumptionKWhRaw").cast("double")
    )
)

StatementMeta(, fa3f41bb-f9fc-4f93-aee4-527020762ac0, 6, Finished, Available, Finished, False)

## 3. Apply Data Quality Rules

Records are validated against the Silver data contract.

### Rejection Rules

- Missing or empty household identifier → `MISSING_HOUSEHOLD`
- Invalid reading timestamp → `INVALID_TIMESTAMP`
- Non-numeric or negative consumption → `INVALID_CONSUMPTION`
- Tariff outside the expected `Std` / `ToU` domain → `INVALID_TARIFF`

Rejected records are retained separately for audit rather than silently discarded.

In [5]:
validated_df = (
    typed_df
    .withColumn(
        "IsMissingHousehold",
        F.col("HouseholdID").isNull() |
        (F.trim(F.col("HouseholdID")) == "")
    )
    .withColumn(
        "IsInvalidTimestamp",
        F.col("ReadingTimestamp").isNull()
    )
    .withColumn(
        "IsInvalidConsumption",
        F.col("ConsumptionKWh").isNull() |
        (F.col("ConsumptionKWh") < 0)
    )
    .withColumn(
        "IsInvalidTariff",
        F.col("TariffType").isNull() |
        (~F.col("TariffType").isin("Std", "ToU"))
    )
)

StatementMeta(, fa3f41bb-f9fc-4f93-aee4-527020762ac0, 7, Finished, Available, Finished, False)

## 4. Separate Valid and Rejected Records

Split the transformed dataset into valid readings and records that fail the Silver validation rules.

In [6]:
validated_df = (
    validated_df
    .withColumn(
        "RejectionReason",
        F.when(
            F.col("IsMissingHousehold"),
            F.lit("MISSING_HOUSEHOLD")
        )
        .when(
            F.col("IsInvalidTimestamp"),
            F.lit("INVALID_TIMESTAMP")
        )
        .when(
            F.col("IsInvalidConsumption"),
            F.lit("INVALID_CONSUMPTION")
        )
        .when(
            F.col("IsInvalidTariff"),
            F.lit("INVALID_TARIFF")
        )
        .otherwise(F.lit(None))
    )
)

StatementMeta(, fa3f41bb-f9fc-4f93-aee4-527020762ac0, 8, Finished, Available, Finished, False)

In [7]:
rejected_df = (
    validated_df
    .filter(F.col("RejectionReason").isNotNull())
)

valid_df = (
    validated_df
    .filter(F.col("RejectionReason").isNull())
)

StatementMeta(, fa3f41bb-f9fc-4f93-aee4-527020762ac0, 9, Finished, Available, Finished, False)

## 5. Deduplicate Business Keys

Valid records are deduplicated using the business key:

`HouseholdID + ReadingTimestamp`

Where duplicate observations exist, one record is retained and the duplicate count is tracked separately for reconciliation.

> Deduplication is treated separately from rejection because duplicate observations can individually satisfy all Silver validation rules.

In [8]:
dedup_window = (
    Window
    .partitionBy(
        "HouseholdID",
        "ReadingTimestamp"
    )
    .orderBy(
        F.col("SourceFileName").asc(),
        F.col("ConsumptionKWh").asc_nulls_last(),
        F.col("TariffType").asc_nulls_last()
    )
)

ranked_df = (
    valid_df
    .withColumn(
        "DuplicateRank",
        F.row_number().over(dedup_window)
    )
)

duplicates_removed_df = (
    ranked_df
    .filter(F.col("DuplicateRank") > 1)
)

silver_base_df = (
    ranked_df
    .filter(F.col("DuplicateRank") == 1)
)

StatementMeta(, fa3f41bb-f9fc-4f93-aee4-527020762ac0, 10, Finished, Available, Finished, False)

## 6. Derive Analytical Attributes

Create reusable date and time attributes from the validated reading timestamp to support downstream profiling and Gold-layer modelling.

In [9]:
silver_df = (
    silver_base_df
    .withColumn(
        "ReadingDate",
        F.to_date("ReadingTimestamp")
    )
    .withColumn(
        "ReadingYear",
        F.year("ReadingTimestamp")
    )
    .withColumn(
        "ReadingMonth",
        F.month("ReadingTimestamp")
    )
    .withColumn(
        "ReadingDay",
        F.dayofmonth("ReadingTimestamp")
    )
    .withColumn(
        "ReadingHour",
        F.hour("ReadingTimestamp")
    )
    .withColumn(
        "DayOfWeek",
        F.date_format("ReadingTimestamp", "EEEE")
    )
)

StatementMeta(, fa3f41bb-f9fc-4f93-aee4-527020762ac0, 11, Finished, Available, Finished, False)

In [10]:
silver_final_df = (
    silver_df
    .select(
        "HouseholdID",
        "TariffType",
        "ReadingTimestamp",
        "ConsumptionKWh",
        "ReadingDate",
        "ReadingYear",
        "ReadingMonth",
        "ReadingDay",
        "ReadingHour",
        "DayOfWeek",
        "SourceFileName",
        "IngestionTimestamp",
        "SourceSystem"
    )
)

StatementMeta(, fa3f41bb-f9fc-4f93-aee4-527020762ac0, 12, Finished, Available, Finished, False)

In [11]:
rejected_output_df = (
    rejected_df
    .select(
        "HouseholdID",
        "TariffType",
        "ReadingTimestampRaw",
        "ConsumptionKWhRaw",
        "SourceFileName",
        "SourceFilePath",
        "IngestionTimestamp",
        "SourceSystem",
        "RejectionReason"
    )
)

StatementMeta(, fa3f41bb-f9fc-4f93-aee4-527020762ac0, 13, Finished, Available, Finished, False)

In [12]:
spark.sql("DROP TABLE IF EXISTS silver.meter_readings")

StatementMeta(, fa3f41bb-f9fc-4f93-aee4-527020762ac0, 14, Finished, Available, Finished, False)

DataFrame[]

## 7. Persist Silver Tables

Persist the validated and rejected datasets as Delta-backed Silver tables.

### Outputs

- `silver.meter_readings` — validated and deduplicated smart-meter readings
- `silver.rejected_meter_readings` — records rejected by Silver validation rules

In [13]:
(
    silver_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.meter_readings")
)

print("silver.meter_readings written successfully.")

StatementMeta(, fa3f41bb-f9fc-4f93-aee4-527020762ac0, 15, Finished, Available, Finished, False)

silver.meter_readings written successfully.


In [14]:
spark.sql("DROP TABLE IF EXISTS silver.rejected_meter_readings")

StatementMeta(, fa3f41bb-f9fc-4f93-aee4-527020762ac0, 16, Finished, Available, Finished, False)

DataFrame[]

In [15]:
(
    rejected_output_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.rejected_meter_readings")
)

print("silver.rejected_meter_readings written successfully.")

StatementMeta(, fa3f41bb-f9fc-4f93-aee4-527020762ac0, 17, Finished, Available, Finished, False)

silver.rejected_meter_readings written successfully.


## 8. Reconciliation

Validate that every Bronze record is accounted for after rejection handling and deduplication.

Reconciliation rule:

`Bronze Rows = Silver Rows + Rejected Rows + Duplicates Removed`

In [16]:
bronze_count = bronze_df.count()
silver_count = spark.table("silver.meter_readings").count()
rejected_count = spark.table("silver.rejected_meter_readings").count()
duplicates_removed_count = duplicates_removed_df.count()

print("FULL SILVER RECONCILIATION")
print("-" * 50)
print(f"Bronze rows:               {bronze_count:,}")
print(f"Rejected rows:             {rejected_count:,}")
print(f"Duplicates removed:        {duplicates_removed_count:,}")
print(f"Silver rows:               {silver_count:,}")
print(
    f"Reconciliation difference: "
    f"{bronze_count - rejected_count - duplicates_removed_count - silver_count:,}"
)

StatementMeta(, fa3f41bb-f9fc-4f93-aee4-527020762ac0, 18, Finished, Available, Finished, False)

FULL SILVER RECONCILIATION
--------------------------------------------------
Bronze rows:               167,932,474
Rejected rows:             5,560
Duplicates removed:        115,453
Silver rows:               167,811,461
Reconciliation difference: 0


In [17]:
print("REJECTION SUMMARY")
print("-" * 50)

spark.table("silver.rejected_meter_readings") \
    .groupBy("RejectionReason") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

StatementMeta(, fa3f41bb-f9fc-4f93-aee4-527020762ac0, 19, Finished, Available, Finished, False)

REJECTION SUMMARY
--------------------------------------------------
+-------------------+-----+
|RejectionReason    |count|
+-------------------+-----+
|INVALID_CONSUMPTION|5560 |
+-------------------+-----+



## 9. Silver Quality Assurance

Run post-transformation quality checks against the persisted Silver dataset to verify that the Silver data contract has been satisfied.

In [18]:
silver_check_df = spark.table("silver.meter_readings")

print("SILVER QA")
print("-" * 50)

duplicate_keys = (
    silver_check_df
    .groupBy(
        "HouseholdID",
        "ReadingTimestamp"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_consumption = (
    silver_check_df
    .filter(
        F.col("ConsumptionKWh").isNull() |
        (F.col("ConsumptionKWh") < 0)
    )
    .count()
)

invalid_tariff = (
    silver_check_df
    .filter(
        ~F.col("TariffType").isin("Std", "ToU")
    )
    .count()
)

print(f"Duplicate business keys: {duplicate_keys:,}")
print(f"Invalid consumption rows: {invalid_consumption:,}")
print(f"Invalid tariff rows:      {invalid_tariff:,}")

StatementMeta(, fa3f41bb-f9fc-4f93-aee4-527020762ac0, 20, Finished, Available, Finished, False)

SILVER QA
--------------------------------------------------
Duplicate business keys: 0
Invalid consumption rows: 0
Invalid tariff rows:      0


## Transformation Summary

The full-scale Silver transformation successfully processed the complete Bronze dataset.

- **167,932,474** Bronze records evaluated
- **5,560** invalid records rejected
- **115,453** duplicate business-key records removed
- **167,811,461** validated Silver records retained
- **0** reconciliation difference
- **0** duplicate business keys remaining
- **0** invalid consumption records remaining
- **0** invalid tariff records remaining

The resulting Silver dataset provides the validated source for downstream profiling, dimensional modelling and Gold-layer aggregation.